# Sentiment Analysis — Method Comparison

This notebook benchmarks three sentiment analysis approaches on the same dataset:

| Method | Module | Type |
|--------|--------|------|
| WordNet / SentiWordNet | `wordnet/` | Lexicon-based |
| SenticNet | `senticnett/` | Knowledge-graph |
| RoBERTa | `rober/` | Transformer |

---

## 1. Setup

In [ ]:
import sys, time
sys.path.insert(0, '..')  # allow imports from project root

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

from wordnet.analyzer   import WordNetAnalyzer
from senticnett.analyzer import SenticNetAnalyzer
from rober.analyzer      import RoBERTaAnalyzer

sns.set_theme(style='whitegrid', palette='muted')
print('Setup complete.')

## 2. Load dataset

In [ ]:
# Replace with your own CSV path if needed.
# Expected columns: 'text' (string) and 'label' (positive/negative/neutral)
DATA_PATH = '../data/sample_reviews.csv'

df = pd.read_csv(DATA_PATH)
print(f'Loaded {len(df):,} rows')
print(df['label'].value_counts())
df.head()

## 3. Initialise analyzers

In [ ]:
analyzers = {
    'WordNet':   WordNetAnalyzer(),
    'SenticNet': SenticNetAnalyzer(),
    'RoBERTa':   RoBERTaAnalyzer(),
}
texts = df['text'].astype(str).tolist()
print('Analyzers ready.')

## 4. Run predictions and time each method

In [ ]:
timings = {}

for name, analyzer in analyzers.items():
    print(f'Running {name}...', end=' ')
    t0 = time.perf_counter()
    results = analyzer.predict_batch(texts)
    elapsed = time.perf_counter() - t0
    timings[name] = elapsed

    df[f'{name}_label'] = [r.label.lower() for r in results]
    print(f'{elapsed:.2f}s  ({elapsed/len(texts)*1000:.1f}ms/sample)')

df.head()

## 5. Accuracy and classification report

In [ ]:
from sklearn.metrics import accuracy_score

print(f'{'Method':<12} Accuracy')
print('-' * 25)
for name in analyzers:
    acc = accuracy_score(df['label'], df[f'{name}_label'])
    print(f'{name:<12} {acc:.1%}')

In [ ]:
for name in analyzers:
    print(f'\n=== {name} ===')
    print(classification_report(df['label'], df[f'{name}_label'], zero_division=0))

## 6. Confusion matrices

In [ ]:
labels = sorted(df['label'].unique())
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, name in zip(axes, analyzers):
    cm = confusion_matrix(df['label'], df[f'{name}_label'], labels=labels)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name)

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Speed comparison

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ms_per_sample = {k: v / len(texts) * 1000 for k, v in timings.items()}
ax.barh(list(ms_per_sample.keys()), list(ms_per_sample.values()), color=['#5B8DCA', '#5BAB87', '#D47A5B'])
ax.set_xlabel('Avg time per sample (ms)')
ax.set_title('Inference speed comparison')
for i, (name, val) in enumerate(ms_per_sample.items()):
    ax.text(val + 0.1, i, f'{val:.1f}ms', va='center', fontsize=10)
plt.tight_layout()
plt.savefig('speed_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Agreement analysis — where do methods disagree?

In [ ]:
# Find samples where all three methods disagree
method_cols = [f'{n}_label' for n in analyzers]
df['all_agree'] = df[method_cols].nunique(axis=1) == 1
df['majority_label'] = df[method_cols].mode(axis=1)[0]

agreement_rate = df['all_agree'].mean()
print(f'All-method agreement: {agreement_rate:.1%}')

print('\nSamples where all three disagree:')
disagreements = df[df[method_cols].nunique(axis=1) == 3]
print(f'  {len(disagreements)} samples ({len(disagreements)/len(df):.1%})')

if len(disagreements) > 0:
    display(disagreements[['text', 'label'] + method_cols].head(10))

## 9. Summary

| Method | Accuracy | Speed | Best use case |
|--------|----------|-------|---------------|
| WordNet | see above | fastest | quick baseline, formal text |
| SenticNet | see above | medium | commonsense, multi-word phrases |
| RoBERTa | see above | slowest | highest accuracy, social media |

**Recommendation:** Use RoBERTa when accuracy matters most; WordNet when you need speed, interpretability, or no internet access.